In [46]:
import pandas as pd
import json

In [64]:
# globals (put into a bash script later)
llm_output_file = "./../output/poster1_20250522_EMA_gpt-4o-mini.json"
annotated_file = "./../data/annotations/EMA_Swissmedic_Drug_Approval - Sheet1.csv"
output_file = "./../evaluation/poster1_20250522_EMA_gpt-4o-mini.csv"

In [62]:
def load_from_json(file_path):
    """
    load data from a JSON file
    """
    with open(file_path, 'r') as file:
        data = json.load(file)
    return pd.DataFrame(data)

def load_from_csv(file_path):
    return pd.read_csv(file_path, encoding='utf-8')


def merge_dfs_with_suffixes(df_llm, df_annotated,
                            key="Marketing_authorisation_number",
                            suffix1="_llm",
                            suffix2="_human",
                            suffix3="_verdict_human",
                            suffix4="_verdict_llm"):

    df1 = df_llm.copy()
    df2 = df_annotated.copy()
    df1[key] = df1[key].str.lower()
    df2[key] = df2[key].str.lower()
    df1 = df1.drop_duplicates(subset=key)
    df2 = df2.drop_duplicates(subset=key)

    common_keys = set(df1[key]).intersection(df2[key])
    df1 = df1[df1[key].isin(common_keys)]
    df2 = df2[df2[key].isin(common_keys)]

    common_cols = sorted(set(df1.columns).intersection(df2.columns))
    df1 = df1[common_cols]
    df2 = df2[common_cols]

    df1 = df1.rename(columns={c: f"{c}{suffix1}" for c in df1.columns})
    df2 = df2.rename(columns={c: f"{c}{suffix2}" for c in df2.columns})

    key1 = key + suffix1
    key2 = key + suffix2
    merged = pd.merge(df1, df2,
                      left_on=key1, 
                      right_on=key2,
                      how="inner")
    


    for col in merged.select_dtypes(include="object"):
        # convert to lowercase for comparison
        merged[col] = merged[col].str.lower()
        # strip trailing and leading whitespace
        merged[col] = merged[col].str.strip()

    for c in common_cols:
        col1 = c + suffix1
        col2 = c + suffix2
        col3 = c + suffix3
        col4 = c + suffix4
        merged[col3] = merged[col1] == merged[col2]
        merged[col4] = ""
    merged = merged[sorted(merged.columns)]

    return merged


In [49]:
df_llm = load_from_json(llm_output_file)
df_llm = df_llm.transpose()
print(len(df_llm))
df_llm.head()

1990


,Origin,Marketing_authorisation_number,Drug,Non_proprietary_name,Drug_class,Pharmaceutical_form,Administration_route,Decision,Current_status,Decision_date,...,Application_date,Nonclinical_pharmacology_species,Nonclinical_pharmacology_strain,Nonclinical_pharmacology_model,Nonclinical_pharmacokinetics_species,Nonclinical_pharmacokinetics_strain,Nonclinical_pharmacokinetics_model,Nonclinical_toxicology_species,Nonclinical_toxicology_strain,Nonclinical_toxicology_model
febseltiq-withdrawal-assessment-report_en.pdf,EMA,EMA/939316/2022,Febseltiq,infigratinib,Small molecule,hard capsule,oral,withdrawn,withdrawn,25.03.2022,...,N/A,"rat, dog",Not reported,Not reported,"rat, dog, monkey",Not reported,Not reported,"rat, dog",Not reported,Not reported
mabcampath-epar-scientific-discussion_en.pdf,EMA,Not reported,MabCampath,alemtuzumab,Biologics,solution for infusion,IV infusion,approved,authorised,01.12.2004,...,Not reported,monkey,cynomolgus monkey,Not reported,monkey,cynomolgus monkey,Not reported,monkey,cynomolgus monkey,Not reported
wainzua-epar-public-assessment-report_en.pdf,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...,Error generating response: Error code: 400 - {...
pluvicto-epar-public-assessment-report_en.pdf,EMA,EMA/871459/2022,Pluvicto,lutetium (177Lu) vipivotide tetraxetan,Biologics,solution for injection/infusion,intravenous,approved,authorised,13.10.2022,...,30.09.2021,"mouse, rat",Not reported,Not reported,"rat, minipig",Not reported,Not reported,"rat, minipig",Not reported,Not reported
uzprovo-epar-public-assessment-report_en.pdf,EMA,EMA/549260/2023,Uzpruvo,ustekinumab,Biologics,Solution for injection,Subcutaneous use,approved,authorised,09.11.2023,...,21.10.2022,monkey,Cynomolgus monkey,Not reported,monkey,Cynomolgus monkey,Not reported,monkey,Cynomolgus monkey,Not reported


In [50]:
df_llm.describe()

,Origin,Marketing_authorisation_number,Drug,Non_proprietary_name,Drug_class,Pharmaceutical_form,Administration_route,Decision,Current_status,Decision_date,...,Application_date,Nonclinical_pharmacology_species,Nonclinical_pharmacology_strain,Nonclinical_pharmacology_model,Nonclinical_pharmacokinetics_species,Nonclinical_pharmacokinetics_strain,Nonclinical_pharmacokinetics_model,Nonclinical_toxicology_species,Nonclinical_toxicology_strain,Nonclinical_toxicology_model
count,1990,1990,1990,1990,1990,1990,1990,1990,1990,1990,...,1990,1990,1990,1990,1990,1990,1990,1990,1990,1990
unique,88,1667,1983,1371,94,573,236,90,90,557,...,1021,331,402,186,222,218,95,215,253,97
top,EMA,<N/A>,SB309,clopidogrel,Small molecule,film-coated tablets,oral,approved,authorised,N/A,...,N/A,Not reported,Not reported,Not reported,Not reported,Not reported,Not reported,Not reported,Not reported,Not reported
freq,1891,121,2,25,1215,211,691,1587,1568,33,...,225,470,1280,1698,699,1493,1817,563,1412,1810


In [63]:
merged_df = merge_dfs_with_suffixes(df_llm, df_annotation)
print(merged_df.shape)

merged_df.head()


(83, 92)


,Administration_route_human,Administration_route_llm,Administration_route_verdict_human,Administration_route_verdict_llm,Application_date_human,Application_date_llm,Application_date_verdict_human,Application_date_verdict_llm,Current_status_human,Current_status_llm,...,Nonclinical_toxicology_strain_verdict_human,Nonclinical_toxicology_strain_verdict_llm,Orphan_drug_status_human,Orphan_drug_status_llm,Orphan_drug_status_verdict_human,Orphan_drug_status_verdict_llm,Pharmaceutical_form_human,Pharmaceutical_form_llm,Pharmaceutical_form_verdict_human,Pharmaceutical_form_verdict_llm
0,oral,iv infusion,False,,24.12.2001,not reported,False,,authorised,authorised,...,False,,no,no,True,,tablet,solution for infusion,False,
1,intravenous injection,"subcutaneous, intravenous",False,,05.04.2024,05.04.2024,True,,authorised,authorised,...,False,,no,no,True,,concentrate for solution for infusion,"solution for injection in pre-filled syringe, ...",False,
2,subcutaneous injection,subcutaneous,False,,12.01.2018,12.01.2018,True,,authorised,authorised,...,False,,no,no,True,,"solution for injection in pre-filled syringe, ...",solution for injection,False,
3,intravenous injection,intravenous,False,,04.12.2015,04.12.2015,True,,authorised,authorised,...,False,,no,no,True,,"solvent for solution for injection, powder for...",lyophilized powder for solution for intravenou...,False,
4,subcutaneous injection,subcutaneous(ly),False,,28.06.2011,28.07.2011,False,,NaN,na,...,False,,no,no,True,,solution for injection,solution for injection,True,


In [65]:
merged_df.to_csv(output_file, index=False, encoding='utf-8')
print(f"Merged DataFrame saved to {output_file}")

Merged DataFrame saved to ./../evaluation/poster1_20250522_EMA_gpt-4o-mini.csv
